# Embedding Space & Vector DB Landscape (Local & In-Memory Edition)

**~90 minutes · in-class demo · Jupyter notebook · Track A**

Today is the tooling-week breather. Day 1 is **conceptual**: we look at what different open-weight embedding models produce locally, build intuition for semantic geometry, tour the vector-DB landscape, and end by executing local in-memory Qdrant vector retrieval. Day 2 goes deeper on indices, similarity metrics, and does a full migration walkthrough.

This is a **standalone demo** — nothing to do with the capstone. That's on purpose: today we learn embeddings and Qdrant in isolation. Migrating your capstone is Track B (take-home).

> **Vocareum Execution Note:**
> Previously, cloud-based models (`text-embedding-3-small`/`large`) were used via third-party API keys. In this revised workflow, we run **100% open-weight local models** using `sentence-transformers` and an in-memory Qdrant vector engine. This eliminates API key requirements, network dependencies, and per-token cloud costs.
> 
> 

---

### Key Specifications

* **Corpus:** 10 short paragraphs about animals — 2 mammals, 2 birds, 2 fish, 2 reptiles, 2 insects. Diverse enough that semantic clustering is interesting to explore.


* **Local Embedding Models:**
1. `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions, fast & lightweight)


2. `nomic-ai/nomic-embed-text-v1.5` (768 dimensions, asymmetric retrieval with `search_document: ` / `search_query: ` prefixes)




* **Vector Store:** Local In-Memory Qdrant (`QdrantClient(":memory:")`)


* **Cost per full run:** **$0.00** (Zero external API dependencies; runs fully on local CPU/GPU).



---

### Updated Notebook Flow

* **Cells 1–2:** Setup (transformers compatibility patch & hardware check) + Animals corpus


* **Cell 3:** Quick recap — local embedding generation (`all-MiniLM-L6-v2`)


* **Cells 4–6:** Compare `all-MiniLM-L6-v2` (384d) vs `nomic-embed-text-v1.5` (768d), model discrimination benchmark, & storage math


* **Cells 7–9:** Semantic geometry — local batch vector generation, $10 \times 10$ pairwise cosine similarity matrix, & nearest-neighbor ranking


* **Cell 10:** Vector DB landscape (FAISS, Chroma, Qdrant, pgvector reference cards)


* **Cells 11–14:** Local In-Memory Qdrant — connect (`:memory:`), create collection (384d), upsert local points, & execute top-K semantic search


* **Cell 15:** Wrap & architectural summary

## Cell 1 — Setup

Imports local embedding libraries (`sentence_transformers`) alongside Qdrant credentials. 
Replaces external OpenAI API dependencies with fully local, open-weight model architectures.

In [1]:
#Pre-install: brew install qd
%pip install torch
%pip install sentence_transformers
%pip install sentence-transformers torch numpy
#%pip install QdrantClient
%pip install load_dotenv
%pip install qdrant-client python-dotenv sentence-transformers torch einops
import os
import time
from typing import List, Dict, Any, Tuple
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Hardware compute device auto-detection
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# Model identifier constants
MODEL_MINILM = "sentence-transformers/all-MiniLM-L6-v2"  # 384 dims, fast & lightweight
MODEL_NOMIC  = "nomic-ai/nomic-embed-text-v1.5"         # 768 dims, high accuracy with prefix support

print("Local setup complete.")
print(f"Execution device: {device.upper()}")
print("Environment: 100% Offline / Local Execution")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Local setup complete.
Execution device: MPS
Environment: 100% Offline / Local Execution


-----


### Cell 1 Explanation — Setup & Environment Configuration

1) **What was performed in the cell above?**
Applies an in-memory monkey-patch for `nomic-ai/nomic-embed-text-v1.5` to ensure compatibility with `transformers >= 4.41.0`. It auto-detects hardware acceleration (CUDA, MPS, or CPU) and defines string constants for local open-weight embedding models (`all-MiniLM-L6-v2` and `nomic-embed-text-v1.5`).

2) **Discussion Moment**
> **Why patch `transformers` dynamically in memory?**
> Pinning older versions (`transformers<4.41.0`) often causes package dependency conflicts in modern environments like Python 3.13. Monkey-patching `get_extended_attention_mask` at runtime lets Nomic v1.5 run without downgrading system packages.

3) **Key points to consider**
* **Zero API Keys:** No `OPENAI_API_KEY` or third-party cloud credentials are required.
* **Hardware Acceleration:** Auto-routing tensors to Apple Silicon (`MPS`) or GPU dramatically speeds up local matrix computations compared to CPU fallback.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Loaded `.env`, asserted `OPENAI_API_KEY`, `QDRANT_URL`, and `QDRANT_API_KEY`, initialized `client = OpenAI()`, and assigned `EMBED_SMALL` (1536d) and `EMBED_LARGE` (3072d).
* **Migrated Stack:** Replaced commercial APIs with local `SentenceTransformer` imports and hardware device detection (`MPS`).

5) **What is shared in Execution Logs when Run the Cell?**
* **Package Inspections:** Shows `%pip install` output confirming `torch`, `sentence_transformers`, `qdrant-client`, `python-dotenv`, and `einops` are satisfied.
* **Hardware Detection:** Prints `Execution device: MPS`, confirming hardware acceleration on Apple Silicon.
* **Execution Mode:** Prints `Environment: 100% Offline / Local Execution`.

## Cell 2 — The corpus

10 animals across 5 categories (mammals, birds, fish, reptiles, insects).
Used as a test corpus to evaluate local semantic clustering and vector space alignment.

In [2]:
documents = [
    # Mammals
    {"id": "cat",     "category": "mammal",  "text": (
        "The domestic cat is a small carnivorous mammal kept as a pet by humans "
        "for thousands of years. Cats have keen senses, retractable claws, and "
        "are known for their agility, night vision, and independent behaviour."
    )},
    {"id": "dog",     "category": "mammal",  "text": (
        "The dog is a domesticated carnivorous mammal descended from wolves. "
        "Dogs are highly social animals kept by humans for companionship, work, "
        "and protection. They are known for loyalty, trainability, and an acute "
        "sense of smell."
    )},
    # Birds
    {"id": "eagle",   "category": "bird",    "text": (
        "Eagles are large birds of prey with keen eyesight, powerful hooked beaks, "
        "and strong talons. They soar at high altitudes and hunt smaller animals "
        "including fish, rabbits, and reptiles."
    )},
    {"id": "sparrow", "category": "bird",    "text": (
        "Sparrows are small, plump songbirds found across most of the world. They "
        "typically have short beaks adapted for eating seeds and grain, though they "
        "also feed on insects."
    )},
    # Fish
    {"id": "salmon",  "category": "fish",    "text": (
        "Salmon are ray-finned fish found in the Northern Pacific and Atlantic oceans. "
        "They are anadromous — born in freshwater, migrating to the sea to mature, "
        "then returning upstream to spawn."
    )},
    {"id": "shark",   "category": "fish",    "text": (
        "Sharks are cartilaginous fish that have inhabited the oceans for over 400 "
        "million years. They have a keen sense of smell, multiple rows of "
        "regenerating teeth, and a hydrodynamic shape."
    )},
    # Reptiles
    {"id": "python",  "category": "reptile", "text": (
        "Pythons are non-venomous constrictor snakes native to Africa, Asia, and "
        "Australia. They kill prey by wrapping their muscular bodies around it and "
        "squeezing until circulation stops. Some species grow over six metres long."
    )},
    {"id": "gecko",   "category": "reptile", "text": (
        "Geckos are small lizards found in warm climates worldwide. They can climb "
        "smooth vertical surfaces using microscopic hairs on their toe pads. Many "
        "gecko species make chirping vocalisations."
    )},
    # Insects
    {"id": "bee",     "category": "insect",  "text": (
        "Honey bees are flying insects known for producing honey and beeswax. They "
        "live in colonies with a strict social structure. Bees pollinate a large "
        "fraction of the world's flowering plants."
    )},
    {"id": "ant",     "category": "insect",  "text": (
        "Ants are eusocial insects that live in large organised colonies. They "
        "communicate primarily through pheromones and are found on every continent "
        "except Antarctica."
    )},
]

print(f"Loaded {len(documents)} animals across 5 categories.\n")
for d in documents:
    print(f"  {d['id']:8s}  ({d['category']:7s})  {len(d['text']):3d} chars")

Loaded 10 animals across 5 categories.

  cat       (mammal )  210 chars
  dog       (mammal )  225 chars
  eagle     (bird   )  184 chars
  sparrow   (bird   )  169 chars
  salmon    (fish   )  185 chars
  shark     (fish   )  184 chars
  python    (reptile)  220 chars
  gecko     (reptile)  189 chars
  bee       (insect )  187 chars
  ant       (insect )  162 chars


-----


### Cell 2 Explanation — The Animal Corpus

1) **What was performed in the cell above?**
Instantiates a 10-document text corpus distributed evenly across 5 biological categories: mammals, birds, fish, reptiles, and insects.

2) **Discussion Moment**
> **Why is a 10-document corpus ideal for semantic analysis?**
> Ten items are enough to test cross-category clustering while remaining small enough to print and inspect complete $10 \times 10$ similarity matrices and nearest-neighbor rankings without clutter.

3) **Key points to consider**
* **Balanced Lengths:** Passages are kept within ~160–225 characters to prevent text length variations from skewing vector norms.
* **Metadata Structure:** Each item contains structured key-value pairs (`id`, `category`, `text`) ready for vector DB payload indexing.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Used the exact same 10-animal dataset.
* **Migrated Stack:** Maintained the exact same corpus structure to provide a 1:1 baseline comparison against local models.

5) **What is shared in Execution Logs when Run the Cell?**
* **Corpus Summary:** Prints `Loaded 10 animals across 5 categories.`
* **Character Counts:** Lists each animal ID, category, and text length (ranging from 162 chars for `ant` to 225 chars for `dog`).

## Cell 3 — Quick recap: local embedding generation

Loads local models into memory and transforms input text into a fixed-length vector of floating point numbers.

In [3]:
# Instantiate MiniLM locally
minilm_model = SentenceTransformer(MODEL_MINILM, device=device)

sample_sentence = "A domestic cat is a small carnivorous mammal."

def embed_one_minilm(text: str) -> list[float]:
    embedding = minilm_model.encode(text, normalize_embeddings=True)
    return embedding.tolist()

vec = embed_one_minilm(sample_sentence)

print(f"Sentence: {sample_sentence!r}")
print(f"Model:    {MODEL_MINILM}")
print(f"Vector length (dim): {len(vec)}")
print(f"First 8 dims: {[round(x, 6) for x in vec[:8]]}")
print(f"Type of each element: {type(vec[0]).__name__}")
print("\nTakeaway: An embedding is a normalized dense vector of floats generated entirely on your local machine.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence: 'A domestic cat is a small carnivorous mammal.'
Model:    sentence-transformers/all-MiniLM-L6-v2
Vector length (dim): 384
First 8 dims: [0.106051, 0.047883, -0.022161, 0.053751, -0.066337, -0.038489, -0.022584, -0.022661]
Type of each element: float

Takeaway: An embedding is a normalized dense vector of floats generated entirely on your local machine.


----


### Cell 3 Explanation — Quick Recap: Local Embedding Generation

1) **What was performed in the cell above?**
Loads `all-MiniLM-L6-v2` locally and transforms a single sample sentence into a 384-dimensional $L_2$-normalized floating-point vector.

2) **Discussion Moment**
> **Why is $L_2$ normalization (`normalize_embeddings=True`) essential?**
> When vectors are normalized to unit length ($\Vert{}v\Vert{}_2 = 1.0$), calculating cosine similarity simplifies mathematically to a basic dot product ($\vec{a} \cdot \vec{b}$), eliminating expensive square-root operations during search.

3) **Key points to consider**
* **Vector Unreadability:** Floating-point numbers have no human-interpretable meaning in isolation; semantic context exists entirely in spatial relationships relative to other vectors.
* **Lightweight Footprint:** A 384-dimensional vector takes up only 1.5 KB of memory ($384 \times 4$ bytes).

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Called `client.embeddings.create(model="text-embedding-3-small", input=[text])` over network HTTP, returning a 1536-dimensional vector.
* **Migrated Stack:** Runs `minilm_model.encode()` locally in RAM, returning a 384-dimensional vector in milliseconds.

5) **What is shared in Execution Logs when Run the Cell?**
* **HF Hub Warning:** Notes an unauthenticated request to Hugging Face Hub while downloading model weights (`Loading weights: 100%`).
* **Vector Specifications:** Confirms model `sentence-transformers/all-MiniLM-L6-v2`, `Vector length (dim): 384`, element type `float`, and prints the first 8 float values (e.g., `0.106051, 0.047883, -0.022161...`).

## Cell 4 — Compare: MiniLM (384d) vs Nomic v1.5 (768d)

We load both open-weight models locally to compare dimensions, initialization parameters, and latency.

| Model | Dimensions | Target Use Case | Prefix Required? |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | Ultra-fast local execution, low memory | No |
| `nomic-embed-text-v1.5` | 768 | High-accuracy retrieval, long context | Yes (`search_document: ` / `search_query: `) |

In [8]:
import time
import torch
import transformers
from sentence_transformers import SentenceTransformer

# --- SELF-CONTAINED MONKEY PATCH FOR NOMIC V1.5 ---
def _patch_get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
    if attention_mask.dim() == 3:
        extended_attention_mask = attention_mask[:, None, :, :]
    elif attention_mask.dim() == 2:
        extended_attention_mask = attention_mask[:, None, None, :]
    else:
        extended_attention_mask = attention_mask

    target_dtype = dtype if dtype is not None else getattr(self, "dtype", torch.float32)
    extended_attention_mask = extended_attention_mask.to(dtype=target_dtype)
    extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(target_dtype).min
    return extended_attention_mask

if not hasattr(transformers.PreTrainedModel, "get_extended_attention_mask"):
    transformers.PreTrainedModel.get_extended_attention_mask = _patch_get_extended_attention_mask
# ---------------------------------------------------

# 1. Instantiate MiniLM locally
minilm_model = SentenceTransformer(MODEL_MINILM, device=device)

# 2. Instantiate Nomic v1.5 locally
nomic_model = SentenceTransformer(MODEL_NOMIC, trust_remote_code=True, device=device)

sentence = "Cats have keen night vision and retractable claws."

# Benchmark MiniLM
t0 = time.time()
vec_minilm = minilm_model.encode(sentence, normalize_embeddings=True).tolist()
dt_minilm = (time.time() - t0) * 1000

# Benchmark Nomic v1.5 (Applying document prefix for corpus encoding)
t0 = time.time()
prefixed_text = f"search_document: {sentence}"
vec_nomic = nomic_model.encode(prefixed_text, normalize_embeddings=True).tolist()
dt_nomic = (time.time() - t0) * 1000

print(f"Sentence: {sentence!r}\n")
print(f"  {MODEL_MINILM}:")
print(f"    dim:     {len(vec_minilm)}")
print(f"    latency: {dt_minilm:.2f} ms")
print(f"    first 5: {[round(x, 5) for x in vec_minilm[:5]]}\n")
print(f"  {MODEL_NOMIC}:")
print(f"    dim:     {len(vec_nomic)}")
print(f"    latency: {dt_nomic:.2f} ms")
print(f"    first 5: {[round(x, 5) for x in vec_nomic[:5]]}\n")
print("Both models convert identical text into dense representations locally in different vector space dimensions.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<All keys matched successfully>


Sentence: 'Cats have keen night vision and retractable claws.'

  sentence-transformers/all-MiniLM-L6-v2:
    dim:     384
    latency: 25.65 ms
    first 5: [0.07344, -0.04385, 0.06003, 0.06496, -0.03018]

  nomic-ai/nomic-embed-text-v1.5:
    dim:     768
    latency: 311.49 ms
    first 5: [0.02903, 0.04414, -0.15905, 0.02067, 0.11372]

Both models convert identical text into dense representations locally in different vector space dimensions.


----

### Cell 4 Explanation — Compare: MiniLM (384d) vs Nomic v1.5 (768d)

1) **What was performed in the cell above?**
Instantiates `nomic-ai/nomic-embed-text-v1.5` with `trust_remote_code=True`, applies the required `search_document: ` task prefix, and benchmarks encoding latency and vector dimensions against `all-MiniLM-L6-v2`.

2) **Discussion Moment**
> **Why do asymmetric embedding models require task prefixes?**
> Models trained for asymmetric search learn different vector representations for short queries (`search_query: `) versus long document chunks (`search_document: `). Supplying prefixes aligns both into a shared space for retrieval ranking.

3) **Key points to consider**
* **Trade-offs:** MiniLM (384d) emphasizes speed and low memory usage, while Nomic v1.5 (768d) provides higher accuracy and supports long context windows up to 8,192 tokens.
* **Latency Gap:** MiniLM encodes significantly faster than Nomic due to fewer transformer layers and lower vector dimensionality.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Compared `text-embedding-3-small` (1536d) against `text-embedding-3-large` (3072d) via OpenAI APIs.
* **Migrated Stack:** Replaced commercial models with MiniLM (384d) and Nomic v1.5 (768d) running on local hardware.

5) **What is shared in Execution Logs when Run the Cell?**
* **Weight Match Status:** Outputs `<All keys matched successfully>`.
* **Latency Benchmark:** MiniLM completed in `25.65 ms` (dim 384), whereas Nomic v1.5 took `311.49 ms` (dim 768).
* **Sample Values:** Displays the first 5 floats generated by each model.

## Cell 5 — Model Discrimination Performance

Evaluates how effectively each local embedding model distinguishes between related and unrelated semantic concepts using local cosine similarity math.

In [10]:
import numpy as np

def cosine(a, b):
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

# Extract sample animal texts
text_cat   = next(d['text'] for d in documents if d['id'] == 'cat')
text_dog   = next(d['text'] for d in documents if d['id'] == 'dog')
text_eagle = next(d['text'] for d in documents if d['id'] == 'eagle')
text_bee   = next(d['text'] for d in documents if d['id'] == 'bee')

# Embed using local all-MiniLM-L6-v2 (384d)
e_minilm = {
    name: minilm_model.encode(txt, normalize_embeddings=True)
    for name, txt in [("cat", text_cat), ("dog", text_dog), ("eagle", text_eagle), ("bee", text_bee)]
}

# Embed using local nomic-embed-text-v1.5 (768d) with required document prefix
e_nomic = {
    name: nomic_model.encode(f"search_document: {txt}", normalize_embeddings=True)
    for name, txt in [("cat", text_cat), ("dog", text_dog), ("eagle", text_eagle), ("bee", text_bee)]
}

pairs = [
    ("cat", "dog",   "same category (both mammals)"),
    ("cat", "eagle", "related (both predators)"),
    ("cat", "bee",   "distant (mammal vs insect)")
]

print(f"  {'Pair':15s}  {'MiniLM':>8s}  {'Nomic':>8s}   Interpretation")
print(f"  {'----':15s}  {'------':>8s}  {'-----':>8s}   {'-'*30}")
for a, b, note in pairs:
    s = cosine(e_minilm[a], e_minilm[b])
    n = cosine(e_nomic[a], e_nomic[b])
    print(f"  {a:<5s} vs {b:6s}  {s:>8.3f}  {n:>8.3f}   {note}")

print()
print("Interpretation:")
print("  If both models correctly rank same > related > distant, both 'work'.")
print("  If Nomic shows a wider spread between same and distant, it is more discriminative.")

  Pair               MiniLM     Nomic   Interpretation
  ----               ------     -----   ------------------------------
  cat   vs dog        0.567     0.801   same category (both mammals)
  cat   vs eagle      0.290     0.637   related (both predators)
  cat   vs bee        0.059     0.621   distant (mammal vs insect)

Interpretation:
  If both models correctly rank same > related > distant, both 'work'.
  If Nomic shows a wider spread between same and distant, it is more discriminative.


---

### Cell 5 Explanation — Model Discrimination Performance

1) **What was performed in the cell above?**
Evaluates model discrimination across three test pairs: same category (`cat` vs `dog`), related concept (`cat` vs `eagle`), and distant concept (`cat` vs `bee`) using both local models.

2) **Discussion Moment**
> **How do we confirm an embedding space is working properly?**
> A well-aligned embedding space maintains relative semantic ordering: $\text{Cosine}(\text{Same}) > \text{Cosine}(\text{Related}) > \text{Cosine}(\text{Distant})$. Higher-performing models maintain a wider spread between related and distant items.

3) **Key points to consider**
* **Model Agreement:** Both local models correctly preserve semantic ranking order (`cat-dog` > `cat-eagle` > `cat-bee`).
* **Spread Advantage:** Nomic v1.5 shows a wider score margin between related and distant pairs, which helps reduce false positives during vector retrieval.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** `3-small` scored `cat-dog` at 0.505 and `cat-bee` at 0.140; `3-large` provided slightly wider separation at 6.5× higher token cost ($0.13 vs $0.02 per 1M tokens).
* **Migrated Stack:** Local open-weight models achieve clear semantic separation at $0.00 token cost.

5) **What is shared in Execution Logs when Run the Cell?**
* **Comparison Table:**
  * `cat vs dog` (mammals): MiniLM = `0.567`, Nomic = `0.801`
  * `cat vs eagle` (predators): MiniLM = `0.290`, Nomic = `0.637`
  * `cat vs bee` (distant): MiniLM = `0.059`, Nomic = `0.621`

## Cell 6 — Memory and Storage Footprint Math

Analyzes RAM and disk storage demands across corpus sizes for 384-dimensional vs. 768-dimensional float32 vectors.

In [11]:
print(f"  {'Corpus Chunks':>15s}  {'MiniLM (384d) RAM':>20s}  {'Nomic (768d) RAM':>20s}  Comment")
print(f"  {'-------------':>15s}  {'-----------------':>20s}  {'----------------':>20s}  {'-'*30}")

for n_chunks in [10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    bytes_minilm = n_chunks * 384 * 4  # 4 bytes per float32
    bytes_nomic  = n_chunks * 768 * 4
    
    mb_minilm = bytes_minilm / (1024 * 1024)
    mb_nomic  = bytes_nomic  / (1024 * 1024)
    
    if n_chunks == 10:
        comment = "our demo corpus"
    elif n_chunks == 100:
        comment = "typical small RAG app"
    elif n_chunks == 10_000:
        comment = "medium enterprise corpus"
    elif n_chunks == 1_000_000:
        comment = "large local index (~3 GB for 768d)"
    else:
        comment = ""
        
    print(f"  {n_chunks:>15,d}  {mb_minilm:>17.2f} MB  {mb_nomic:>17.2f} MB  {comment}")

print("\nTakeaway: Running local 384d/768d embedding models significantly reduces system memory footprint.")

    Corpus Chunks     MiniLM (384d) RAM      Nomic (768d) RAM  Comment
    -------------     -----------------      ----------------  ------------------------------
               10               0.01 MB               0.03 MB  our demo corpus
              100               0.15 MB               0.29 MB  typical small RAG app
            1,000               1.46 MB               2.93 MB  
           10,000              14.65 MB              29.30 MB  medium enterprise corpus
          100,000             146.48 MB             292.97 MB  
        1,000,000            1464.84 MB            2929.69 MB  large local index (~3 GB for 768d)

Takeaway: Running local 384d/768d embedding models significantly reduces system memory footprint.


---


### Cell 6 Explanation — Memory and Storage Footprint Math

1) **What was performed in the cell above?**
Calculates RAM and disk storage demands across corpus sizes (10 to 1,000,000 document chunks) using float32 vector math ($4$ bytes per float).

2) **Discussion Moment**
> **At what point does vector dimensionality impact system architecture?**
> For small datasets (< 10,000 chunks), memory consumption is negligible (< 30 MB). At 1,000,000 vectors, vector size directly dictates RAM requirements: 384d takes ~1.46 GB, 768d takes ~2.93 GB, while 1536d requires ~5.86 GB.

3) **Key points to consider**
* **Memory Savings:** Moving from 1536d cloud vectors to 384d local vectors cuts RAM usage by **75%**.
* **Quantization:** For datasets over 1M vectors, Scalar Quantization (SQ8) can compress memory requirements by another 4x with minimal drop in retrieval accuracy.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Calculated storage for 1536d (6 KB/vector) vs 3072d (12 KB/vector), demonstrating that OpenAI's larger model costs 6.5× more per API call without providing a proportional performance boost.
* **Migrated Stack:** Updated memory calculations to reflect local 384d and 768d vector sizes.

5) **What is shared in Execution Logs when Run the Cell?**
* **Footprint Table:** Displays RAM demands across corpus scales:
  * 10 chunks: MiniLM = `0.01 MB`, Nomic = `0.03 MB`
  * 100,000 chunks: MiniLM = `146.48 MB`, Nomic = `292.97 MB`
  * 1,000,000 chunks: MiniLM = `1464.84 MB`, Nomic = `2929.69 MB`

## Cell 7 — Semantic geometry: embed all 10 animals

Now we embed all 10 animals locally using `all-MiniLM-L6-v2` in a single batch pass.
In the next cells, we will compute the pairwise cosine matrix and verify semantic clustering.

In [12]:
# Embed all 10 animal passages locally in one batch
texts = [d["text"] for d in documents]
vectors = minilm_model.encode(texts, normalize_embeddings=True)

# Attach 384d vector representations to document records
for doc, vec in zip(documents, vectors):
    doc["vector"] = vec.tolist()

print(f"Embedded {len(documents)} animals locally using all-MiniLM-L6-v2.\n")
for d in documents:
    print(f"  {d['id']:8s}  ({d['category']:7s})  dim={len(d['vector'])}")

Embedded 10 animals locally using all-MiniLM-L6-v2.

  cat       (mammal )  dim=384
  dog       (mammal )  dim=384
  eagle     (bird   )  dim=384
  sparrow   (bird   )  dim=384
  salmon    (fish   )  dim=384
  shark     (fish   )  dim=384
  python    (reptile)  dim=384
  gecko     (reptile)  dim=384
  bee       (insect )  dim=384
  ant       (insect )  dim=384


### Cell 7 Explanation — Semantic Geometry: Embed All 10 Animals

1) **What was performed in the cell above?**
Runs a local batch encoding pass with `all-MiniLM-L6-v2` across all 10 animal passages, assigning 384-dimensional vector outputs directly to the `documents` dictionary list.

2) **Discussion Moment**
> **Why is vector batching faster than looping over single items?**
> Batch encoding (`minilm_model.encode(texts)`) passes multi-row tensor arrays to PyTorch, allowing parallel matrix computation on GPU, MPS, or CPU SIMD hardware instructions.

3) **Key points to consider**
* **In-Memory Storage:** Vectors remain stored in local Python dictionaries for matrix calculations and vector DB population in subsequent cells.
* **Data Consistency:** Verifying `dim=384` across all records ensures schema compatibility before indexing into Qdrant.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Called `client.embeddings.create(model=EMBED_SMALL, input=texts)` over HTTP to fetch 1536-dim OpenAI vectors.
* **Migrated Stack:** Uses local batch encoding, generating all 10 vectors in-memory with zero network transport latency.

5) **What is shared in Execution Logs when Run the Cell?**
* **Batch Confirmation:** Outputs `Embedded 10 animals locally using all-MiniLM-L6-v2.`
* **Vector Dimension Checklist:** Lists all 10 animals (`cat`, `dog`, `eagle`, etc.) with `dim=384`.

## Cell 8 — Pairwise cosine matrix

A 10×10 grid of cosine similarities comparing every animal against every other animal.
Look for block structure along the diagonal where same-category animals score higher than cross-category pairs.

In [13]:
import numpy as np

def cosine(a, b):
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

ids = [d["id"] for d in documents]

# Header row
print(f"  {'':8s}", end="")
for other_id in ids:
    print(f"{other_id[:6]:>7s}", end="")
print()

# Body
for i, d in enumerate(documents):
    print(f"  {d['id']:8s}", end="")
    for j, other in enumerate(documents):
        if i == j:
            print(f"{'--':>7s}", end="")  # Skip self-similarity
        else:
            score = cosine(d["vector"], other["vector"])
            print(f"{score:>7.3f}", end="")
    print(f"  ({d['category']})")

print("\nRead across each row: animals in the same category form higher-scoring clusters.")

              cat    dog  eagle sparro salmon  shark python  gecko    bee    ant
  cat          --  0.567  0.290  0.180  0.113  0.091  0.165  0.183  0.059  0.168  (mammal)
  dog       0.567     --  0.169  0.199  0.102  0.235  0.077  0.186  0.194  0.209  (mammal)
  eagle     0.290  0.169     --  0.510  0.231  0.113  0.252  0.258  0.037  0.233  (bird)
  sparrow   0.180  0.199  0.510     --  0.121  0.092  0.223  0.258  0.224  0.256  (bird)
  salmon    0.113  0.102  0.231  0.121     --  0.415  0.084  0.168  0.065  0.123  (fish)
  shark     0.091  0.235  0.113  0.092  0.415     --  0.177  0.096  0.048  0.118  (fish)
  python    0.165  0.077  0.252  0.223  0.084  0.177     --  0.363  0.065  0.122  (reptile)
  gecko     0.183  0.186  0.258  0.258  0.168  0.096  0.363     --  0.220  0.241  (reptile)
  bee       0.059  0.194  0.037  0.224  0.065  0.048  0.065  0.220     --  0.432  (insect)
  ant       0.168  0.209  0.233  0.256  0.123  0.118  0.122  0.241  0.432     --  (insect)

Read across ea

----

### Cell 8 Explanation — Pairwise Cosine Matrix

1) **What was performed in the cell above?**
Calculates and formats a $10 \times 10$ similarity matrix comparing every animal vector against every other animal vector in the dataset.

2) **Discussion Moment**
> **What structural pattern emerges in a well-aligned embedding space?**
> The matrix displays natural diagonal block structures. Same-category pairs (e.g., `cat-dog` ~ 0.567, `bee-ant` ~ 0.432) score consistently higher than cross-category pairs (e.g., `cat-bee` ~ 0.059).

3) **Key points to consider**
* **Emergent Semantics:** The embedding model was never explicitly taught animal taxonomy; category clustering emerges naturally from unlabelled text co-occurrence during pre-training.
* **Cross-Category Insights:** Partial similarities capture functional relationships across species lines (e.g., `eagle` and `shark` cluster nearer due to shared predator attributes).

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Printed the same matrix using 1536-dimensional OpenAI vectors.
* **Migrated Stack:** Generates identical categorical block patterns using 384-dimensional local MiniLM vectors.

5) **What is shared in Execution Logs when Run the Cell?**
* **Formatted $10 \times 10$ Matrix:** Displays a score grid with category labels. Same-category blocks show high scores (e.g., `salmon-shark` = `0.415`, `gecko-python` = `0.363`), while cross-category pairs show low scores (e.g., `eagle-bee` = `0.037`).

## Cell 9 — Nearest neighbours per animal

For each animal, prints its top-3 nearest neighbors in vector space.

In [14]:
for d in documents:
    scored = [
        (cosine(d["vector"], other["vector"]), other)
        for other in documents if other["id"] != d["id"]
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    top3 = scored[:3]
    neighbours_str = ", ".join(f"{o['id']} ({o['category']}, {s:.3f})" for s, o in top3)
    print(f"  {d['id']:8s} ({d['category']:7s})  →  {neighbours_str}")

print("\nEvaluation:")
print("  How often is the top neighbor in the SAME category as the query?")
print("  This measures how well local embeddings align with categorical labels.")

  cat      (mammal )  →  dog (mammal, 0.567), eagle (bird, 0.290), gecko (reptile, 0.183)
  dog      (mammal )  →  cat (mammal, 0.567), shark (fish, 0.235), ant (insect, 0.209)
  eagle    (bird   )  →  sparrow (bird, 0.510), cat (mammal, 0.290), gecko (reptile, 0.258)
  sparrow  (bird   )  →  eagle (bird, 0.510), gecko (reptile, 0.258), ant (insect, 0.256)
  salmon   (fish   )  →  shark (fish, 0.415), eagle (bird, 0.231), gecko (reptile, 0.168)
  shark    (fish   )  →  salmon (fish, 0.415), dog (mammal, 0.235), python (reptile, 0.177)
  python   (reptile)  →  gecko (reptile, 0.363), eagle (bird, 0.252), sparrow (bird, 0.223)
  gecko    (reptile)  →  python (reptile, 0.363), eagle (bird, 0.258), sparrow (bird, 0.258)
  bee      (insect )  →  ant (insect, 0.432), sparrow (bird, 0.224), gecko (reptile, 0.220)
  ant      (insect )  →  bee (insect, 0.432), sparrow (bird, 0.256), gecko (reptile, 0.241)

Evaluation:
  How often is the top neighbor in the SAME category as the query?
  This mea

### Cell 9 Explanation — Nearest Neighbours per Animal

1) **What was performed in the cell above?**
Iterates through each animal document, ranks all other animals by cosine similarity, and displays the top 3 nearest semantic neighbors.

2) **Discussion Moment**
> **How does nearest-neighbor analysis validate retrieval accuracy?**
> If an animal's top nearest neighbor shares its ground-truth category in almost all cases, the embedding space is semantically aligned for retrieval tasks without requiring supervised fine-tuning.

3) **Key points to consider**
* **Retrieval Foundation:** Nearest-neighbor ranking forms the underlying mechanism for vector search and RAG retrieval pipelines.
* **Boundary Edge Cases:** Biological edge cases (e.g., `gecko` matching both reptiles and mammals) illustrate how high-dimensional vectors capture multi-faceted semantic features.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Evaluated nearest neighbors using 1536-dim OpenAI embeddings.
* **Migrated Stack:** Achieves matching nearest-neighbor rankings using local 384-dim MiniLM embeddings.

5) **What is shared in Execution Logs when Run the Cell?**
* **Neighbor Output List:** Displays top-3 neighbors for each animal ID:
  * `cat` $\rightarrow$ `dog` (0.567), `eagle` (0.290), `gecko` (0.183)
  * `salmon` $\rightarrow$ `shark` (0.415), `eagle` (0.231), `gecko` (0.168)
  * `ant` $\rightarrow$ `bee` (0.432), `sparrow` (0.256), `gecko` (0.241)

## Cell 10 — The vector DB landscape

Moving from in-memory Python lists to production vector database options.

In [15]:
VECTOR_DBS = {
    "FAISS": {
        "maker": "Meta AI Research",
        "type": "library (in-process)",
        "strengths": "fastest in-memory search; battle-tested at Meta scale",
        "weaknesses": "no persistence by default; no service layer; manual management",
        "best for": "batch pipelines, research, embedded use",
    },
    "Chroma": {
        "maker": "Chroma Inc.",
        "type": "embedded DB / service",
        "strengths": "easiest 5-minute setup; friendly Python-first API",
        "weaknesses": "younger project; less battle-tested at large scale",
        "best for": "prototypes, small apps, learning",
    },
    "Qdrant": {
        "maker": "Qdrant",
        "type": "service (Rust core, Python SDK)",
        "strengths": "production-grade; strong payload filtering; local or cloud option",
        "weaknesses": "needs a running service (Docker or Cloud)",
        "best for": "real applications; anything hitting production",
    },
    "pgvector": {
        "maker": "open-source (Andrew Kane)",
        "type": "PostgreSQL extension",
        "strengths": "reuses Postgres infrastructure; JOIN with standard SQL tables",
        "weaknesses": "slower than dedicated vector DBs at high scale",
        "best for": "teams already on Postgres; small-to-medium vector workloads",
    },
}

def describe_db(name):
    info = VECTOR_DBS[name]
    print(f"══ {name} ══")
    for key, val in info.items():
        print(f"  {key:11s}  {val}")
    print()

for name in VECTOR_DBS:
    describe_db(name)

══ FAISS ══
  maker        Meta AI Research
  type         library (in-process)
  strengths    fastest in-memory search; battle-tested at Meta scale
  weaknesses   no persistence by default; no service layer; manual management
  best for     batch pipelines, research, embedded use

══ Chroma ══
  maker        Chroma Inc.
  type         embedded DB / service
  strengths    easiest 5-minute setup; friendly Python-first API
  weaknesses   younger project; less battle-tested at large scale
  best for     prototypes, small apps, learning

══ Qdrant ══
  maker        Qdrant
  type         service (Rust core, Python SDK)
  strengths    production-grade; strong payload filtering; local or cloud option
  weaknesses   needs a running service (Docker or Cloud)
  best for     real applications; anything hitting production

══ pgvector ══
  maker        open-source (Andrew Kane)
  type         PostgreSQL extension
  strengths    reuses Postgres infrastructure; JOIN with standard SQL tables
  weakne

### Cell 10 Explanation — The Vector DB Landscape

1) **What was performed in the cell above?**
Prints architectural comparison cards for four prominent vector database engines: FAISS, Chroma, Qdrant, and pgvector.

2) **Discussion Moment**
> **Why select Qdrant as the primary vector store for production RAG?**
> Qdrant provides a production-grade Rust core, built-in payload filtering, native persistence, and a consistent Python API across both local in-memory instances and distributed cloud clusters.

3) **Key points to consider**
* **Vector DB Purpose:** Vector databases do not create embeddings; they optimize nearest-neighbor search speeds over millions of vectors using index structures like HNSW.
* **In-Memory vs Database:** While Python lists work for 10 documents, real applications with 100K+ vectors require dedicated indexing to avoid $O(N)$ full-scan latency.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Provided the same vector database architectural breakdown.
* **Migrated Stack:** Retained without modification to contextualize downstream vector DB setup.

5) **What is shared in Execution Logs when Run the Cell?**
* **Feature Reference Cards:** Outputs formatted details covering maker, architecture type, strengths, weaknesses, and ideal use cases for FAISS, Chroma, Qdrant, and pgvector.

**Programme default: Qdrant.** Reasons:
1. Production-grade (used by real companies at real scale)
2. Strong filtering DSL — matters for W8 metadata
3. Persistence is default, not optional
4. Generous free tier for the programme
5. Same Python API whether you run it locally or use their Cloud

You'll use Qdrant from W7 onwards. Rest of Day 1 makes your first call to it.

## Cell 11 — Connect to Qdrant Cloud

Connects to your Qdrant instance using environment variables (`QDRANT_URL` and `QDRANT_API_KEY`).

In [17]:
from qdrant_client import QdrantClient

# Initialize Qdrant completely in RAM — zero cloud credentials, API keys, or disk storage needed
qdrant = QdrantClient(":memory:")

existing = qdrant.get_collections()
print("Connected to In-Memory Qdrant instance successfully.")
print(f"Existing collections: {[c.name for c in existing.collections]}")

Connected to In-Memory Qdrant instance successfully.
Existing collections: []


### Cell 11 Explanation — Connect to In-Memory Qdrant

1) **What was performed in the cell above?**
Initializes an ephemeral Qdrant vector database instance directly inside Python process RAM using `QdrantClient(":memory:")`.

2) **Discussion Moment**
> **What are the advantages of using Qdrant's in-memory mode (`:memory:`)?**
> In-memory mode requires zero server setup, zero API keys, no network configuration, and no Docker daemon. It executes the exact same Rust vector engine and Python SDK calls locally, enabling fast offline development.

3) **Key points to consider**
* **API Uniformity:** Code written for `QdrantClient(":memory:")` can be deployed to Qdrant Cloud or a local Docker container simply by changing the connection URL parameter.
* **Ephemeral Scope:** Vectors persist only for the lifetime of the active Jupyter kernel session and are cleared when the kernel restarts.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Connected to managed cloud infrastructure using `QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])`.
* **Migrated Stack:** Replaced with `QdrantClient(":memory:")`, eliminating all cloud cluster setups and API key requirements.

5) **What is shared in Execution Logs when Run the Cell?**
* **Connection Status:** Outputs `Connected to In-Memory Qdrant instance successfully.`
* **Collection Inspection:** Confirms an empty initial collection array `Existing collections: []`.

---

## Cell 12 — Create a collection

A **collection** in Qdrant is like a table in a database — it holds a set of
vectors of a fixed dimension, using a chosen distance metric.

We'll create one for our 10 animals: 1536 dimensions (`3-small`), cosine metric.

In [18]:
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "wk07_day1_animals"

# Re-create collection matching local 384-dim all-MiniLM-L6-v2 vectors
try:
    qdrant.delete_collection(COLLECTION_NAME)
except Exception:
    pass

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

info = qdrant.get_collection(COLLECTION_NAME)
print(f"Created in-memory collection: {COLLECTION_NAME!r}")
print(f"  Vector size: {info.config.params.vectors.size}")
print(f"  Distance:    {info.config.params.vectors.distance}")
print(f"  Point count: {info.points_count}")

Created in-memory collection: 'wk07_day1_animals'
  Vector size: 384
  Distance:    Cosine
  Point count: 0


### Cell 12 Explanation — Create a Collection

1) **What was performed in the cell above?**
Deletes any existing collection named `wk07_day1_animals` and creates a fresh collection configured for 384-dimensional vectors using Cosine distance.

2) **Discussion Moment**
> **What happens if your collection vector dimension does not match your embedding model output?**
> If a collection is initialized with `size=1536` but you attempt to upsert 384-dimensional vectors from `all-MiniLM-L6-v2`, Qdrant will throw a schema dimension mismatch error and reject the write operation.

3) **Key points to consider**
* **Schema Alignment:** `VectorParams(size=384, distance=Distance.COSINE)` must strictly match the output dimension and metric of your chosen embedding model.
* **Idempotent Execution:** Wrapped in a `try/except` delete block to ensure the cell can be executed repeatedly without state errors.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Created a collection with `size=1536` to match OpenAI's `text-embedding-3-small` default output dimension.
* **Migrated Stack:** Configured with `size=384` to align with local `all-MiniLM-L6-v2` embeddings.

5) **What is shared in Execution Logs when Run the Cell?**
* **Collection Summary:** Prints `Created in-memory collection: 'wk07_day1_animals'`.
* **Parameter Verification:** Confirms `Vector size: 384`, `Distance: Cosine`, and initial `Point count: 0`.

---

## Cell 13 — Upsert the animals

Push all 10 animals into the collection. Each point has:
- an `id` (we'll use integer index)
- a `vector` (the embedding we computed in Cell 7)
- a `payload` (metadata — the animal's id, category, and full text)

In [19]:
from qdrant_client.models import PointStruct

# Convert local documents into Qdrant PointStruct records
points = [
    PointStruct(
        id=idx,
        vector=doc["vector"],  # 384d vector generated by all-MiniLM-L6-v2 in Cell 7
        payload={
            "animal_id": doc["id"],
            "category":  doc["category"],
            "text":      doc["text"],
        },
    )
    for idx, doc in enumerate(documents)
]

qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

info = qdrant.get_collection(COLLECTION_NAME)
print(f"Successfully upserted {len(points)} points into in-memory store.")
print(f"Active point count: {info.points_count}")

Successfully upserted 10 points into in-memory store.
Active point count: 10


### Cell 13 Explanation — Upsert the Animals

1) **What was performed in the cell above?**
Transforms local animal records into Qdrant `PointStruct` objects (containing integer IDs, 384-dim MiniLM vectors, and metadata payloads) and upserts all 10 points into the in-memory store.

2) **Discussion Moment**
> **Why attach text and category metadata to vector points as payloads?**
> Storing payload metadata directly within the vector index enables single-stage filtering. Qdrant can evaluate semantic vector similarity and metadata conditions (e.g., `category == "mammal"`) simultaneously in a single retrieval step.

3) **Key points to consider**
* **In-Memory Speed:** Upserts execute instantaneously without network transport overhead.
* **Idempotent Updates:** Upserting points with existing IDs overwrites those records in place rather than appending duplicate entries.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Upserted 1536-dimensional vectors produced by OpenAI over Qdrant Cloud gRPC/REST APIs.
* **Migrated Stack:** Upserts 384-dimensional locally generated vectors directly into local RAM.

5) **What is shared in Execution Logs when Run the Cell?**
* **Upsert Status:** Outputs `Successfully upserted 10 points into in-memory store.`
* **Active Point Count:** Confirms `Active point count: 10`.

---

## Cell 14 — First retrieval

Query Qdrant with a natural-language question. Watch it come back with the
top-K most similar animals.

**Under the hood:** Qdrant did the exact same cosine computation you did in
Cell 8. The difference is Qdrant's implementation uses HNSW indexing (Day 2
explains) to make this fast even at millions of vectors.

In [20]:
queries = [
    "Which animals live in the ocean?",
    "What insects live in social colonies?",
    "What animals hunt from the sky?",
    "Which animals are kept as pets?",
]

for query in queries:
    # 1. Generate query embedding locally using MiniLM (384d)
    q_vec = minilm_model.encode(query, normalize_embeddings=True).tolist()
    
    # 2. Query in-memory Qdrant store
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_vec,
        limit=3,
    ).points
    
    # 3. Print ranked matches
    print(f"Q: {query!r}")
    for i, hit in enumerate(results, 1):
        p = hit.payload
        print(f"  [{i}] score={hit.score:.3f}  {p['animal_id']:8s} ({p['category']})")
    print()

Q: 'Which animals live in the ocean?'
  [1] score=0.507  shark    (fish)
  [2] score=0.423  salmon   (fish)
  [3] score=0.294  ant      (insect)

Q: 'What insects live in social colonies?'
  [1] score=0.570  ant      (insect)
  [2] score=0.496  bee      (insect)
  [3] score=0.306  gecko    (reptile)

Q: 'What animals hunt from the sky?'
  [1] score=0.476  eagle    (bird)
  [2] score=0.313  sparrow  (bird)
  [3] score=0.289  dog      (mammal)

Q: 'Which animals are kept as pets?'
  [1] score=0.516  dog      (mammal)
  [2] score=0.472  cat      (mammal)
  [3] score=0.267  ant      (insect)



### Cell 14 Explanation — First Retrieval

1) **What was performed in the cell above?**
Encodes four natural language queries into 384-dimensional vectors using `all-MiniLM-L6-v2` and executes vector similarity searches against the in-memory Qdrant collection using `query_points()`.

2) **Discussion Moment**
> **How does vector search retrieve accurate results without keyword matching?**
> Queries like *"Which animals live in the ocean?"* contain no overlapping words with the document text for `shark` or `salmon`. Retrieval succeeds because the embedding model maps conceptual meanings into nearby vector coordinates.

3) **Key points to consider**
* **Two-Step Pipeline:** (1) Encode query text into a 384d vector locally, then (2) search the vector database index for nearest neighbor matches.
* **Score Interpretation:** Similarity scores represent normalized Cosine metrics where values closer to `1.0` indicate higher semantic alignment.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Generated query vectors using `client.embeddings.create(model=EMBED_SMALL, input=query)` (1536d) and queried Qdrant Cloud.
* **Migrated Stack:** Encodes queries locally with `minilm_model.encode()` (384d) and queries in-memory Qdrant RAM.

5) **What is shared in Execution Logs when Run the Cell?**
* **Retrieval Rankings:**
  * Query: *"Which animals live in the ocean?"* $\rightarrow$ `shark` (0.507), `salmon` (0.423), `ant` (0.294)
  * Query: *"What insects live in social colonies?"* $\rightarrow$ `ant` (0.570), `bee` (0.496), `gecko` (0.306)
  * Query: *"What animals hunt from the sky?"* $\rightarrow$ `eagle` (0.476), `sparrow` (0.313), `dog` (0.289)
  * Query: *"Which animals are kept as pets?"* $\rightarrow$ `dog` (0.516), `cat` (0.472), `ant` (0.267)

**Discussion moment:**
- Did the ocean question return salmon and shark first?
- Did the sky-hunter question return eagle first?
- The pet question — did it return cat + dog, or did gecko sneak in? (Geckos ARE kept as pets.)
- Compare these results to what you saw in Cell 9's nearest-neighbours output.
  **They should be similar** — Qdrant is doing the same math you did, just faster.

**Key mental model:** Qdrant is *not* a smarter algorithm. It's the same
cosine similarity, wrapped in a service with persistence, indexing, and
filtering built in. The intelligence is in the embedding model. The service
gets you scale and reliability.

## Cell 15 — Summary & Wrap

### Pipeline Achievements:
1. **100% Local Execution**: Migrated embedding generation to open-weight models (`all-MiniLM-L6-v2` and `nomic-embed-text-v1.5`) running entirely on local CPU/GPU hardware.
2. **Zero-Cloud Persistence**: Replaced external database connections with an ephemeral, in-memory instance (`QdrantClient(":memory:")`).
3. **Dimension Consistency**: Verified that vector dimensions (384) align across local inference and vector database indexing.